# مرشّح النطاق (Band-Pass Filter)

**Dataset**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**Channels**: P4, Cz, F8, T7  
**Sampling rate**: 200 Hz  
**Subject 7 (only one subject for speed)

---

## What this notebook does

A band-pass filter combines a high-pass and a low-pass filter into **one operation**. It keeps only the frequencies between 1 and 40 Hz, which covers all the important brain waves:

| Band | Frequency | Associated with |
|------|-----------|-----------------|
| Delta | 1 to 4 Hz | Deep sleep |
| Theta | 4 to 8 Hz | Drowsiness, memory |
| Alpha | 8 to 13 Hz | Relaxation (eyes closed) |
| Beta | 13 to 30 Hz | Active thinking, focus |

## What you should expect to see

The filtered signal should be:
- **Centered around zero** (no DC offset, like the high-pass)
- **Smooth** (no high-frequency noise, like the low-pass)
- **Clean enough to see artifacts** like eye blinks and chewing, which were hidden by noise before

## Why not just use high-pass + low-pass separately?

You can, and the result is the same. But the band-pass filter does it in **one function call** and is more efficient. We show both approaches so you understand what happens under the hood.

## 1. Install dependencies

In [ ]:
!pip install scipy numpy plotly wfdb

## 2. Clone the resources repo and download one subject

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')

In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 7

## 3. Load the EEG signal

We load subject 7, experiment 1, session 2, channel **P4** (parietal region).

In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=7, experiment=1, session=2
)
channel_data = eeg_data[:, 0]  # P4 channel
fs = 200  # Sampling rate (Hz)

print(f'Channels: {ch_names}')
print(f'Signal length: {len(channel_data)} samples ({len(channel_data)/fs:.1f} seconds)')

## 4. Apply the band-pass filter (1 to 40 Hz)

We use `signal.butter` with `btype='band'` and pass both cutoff frequencies as a list. The `filtfilt` function ensures zero phase delay.

In [ ]:
from scipy import signal

def butter_bandpass_filter(data, lowcut, highcut, fs, order=4):
    nyq = 0.5 * fs  # Nyquist = 100 Hz
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype='band', analog=False)
    return signal.filtfilt(b, a, data)

filtered_bp = butter_bandpass_filter(
    channel_data, lowcut=1.0, highcut=40.0, fs=fs
)
print(f'Band-pass filter applied: {1.0} to {40.0} Hz, order {4}')
print(f'Nyquist frequency: {0.5*fs} Hz')

## 5. Interactive plot: raw vs band-pass filtered

**What to look for:**
- The raw signal (top) has both drift and noise
- The filtered signal (bottom) is centered around zero AND smooth
- You may notice **artifacts** (sudden spikes) in the filtered signal that were hidden by noise before. These could be eye blinks or muscle movements. We will deal with artifacts in a later chapter.

Use the zoom tool to inspect specific time ranges. Hover over the signal to see exact values.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

n_plot = min(5000, len(channel_data))
t_sec = timestamps[:n_plot] / 1000.0

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Raw EEG (P4)', 'Band-pass filtered (1 to 40 Hz)'))

fig.add_trace(go.Scatter(x=t_sec, y=channel_data[:n_plot],
                         name='Raw', line=dict(color='gray', width=0.5)),
               row=1, col=1)
fig.add_trace(go.Scatter(x=t_sec, y=filtered_bp[:n_plot],
                         name='Band-pass', line=dict(color='blue', width=0.5)),
               row=2, col=1)

fig.update_layout(height=600, title_text='Band-Pass Filter: Raw vs Filtered',
                  xaxis2_title='Time (s)', yaxis_title='EEG (uV)',
                  yaxis2_title='EEG (uV)')
fig.show()

## 6. Compare all three filters

Let's overlay the raw signal and all three filtered versions to see the progression:
1. **Raw** (gray): drift + noise
2. **High-pass only** (green): drift removed, noise remains
3. **High-pass + low-pass** (red): drift removed, noise removed
4. **Band-pass** (blue): same as HP+LP but in one step

Notice how the red and blue lines overlap almost perfectly. This confirms that band-pass = high-pass + low-pass.

In [ ]:
# Apply all three filters for comparison
def butter_highpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs
    b, a = signal.butter(order, cutoff / nyq, btype='high', analog=False)
    return signal.filtfilt(b, a, data)

def butter_lowpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs
    b, a = signal.butter(order, cutoff / nyq, btype='low', analog=False)
    return signal.filtfilt(b, a, data)

hp = butter_highpass_filter(channel_data, 1.0, fs)
hp_lp = butter_lowpass_filter(hp, 40.0, fs)
bp = butter_bandpass_filter(channel_data, 1.0, 40.0, fs)

n_plot = min(3000, len(channel_data))
t_sec = timestamps[:n_plot] / 1000.0

fig = go.Figure()
fig.add_trace(go.Scatter(x=t_sec, y=channel_data[:n_plot], name='Raw',
                         line=dict(color='gray', width=0.5), opacity=0.5))
fig.add_trace(go.Scatter(x=t_sec, y=hp[:n_plot], name='High-pass only',
                         line=dict(color='green', width=0.5)))
fig.add_trace(go.Scatter(x=t_sec, y=hp_lp[:n_plot], name='HP + LP',
                         line=dict(color='red', width=0.5)))
fig.add_trace(go.Scatter(x=t_sec, y=bp[:n_plot], name='Band-pass',
                         line=dict(color='blue', width=0.5)))

fig.update_layout(height=500, title_text='All filters compared (zoom in to see differences)',
                  xaxis_title='Time (s)', yaxis_title='EEG (uV)')
fig.show()

## 7. What did we learn?

- The band-pass filter (1 to 40 Hz) combines high-pass and low-pass in **one step**
- The result is identical to applying them sequentially
- The filtered signal is clean enough to **see artifacts** (eye blinks, muscle movements)
- This is the **first real preprocessing step** in any EEG analysis pipeline
- The next chapter covers **smoothing filters** (moving average, Gaussian) which further reduce noise